#Creating the pipeline within Bronze_layer

In [0]:
display(dbutils.fs.ls("/Volumes/ecommerce/bronze/raw_files/"))

##DDL of pipeline_logs table(delta table)

In [0]:
%sql
DROP TABLE IF EXISTS ecommerce.bronze.pipeline_logs;

In [0]:
%sql
CREATE TABLE ecommerce.bronze.pipeline_logs (
    pipeline_name STRING,
    table_name STRING,
    stage STRING,
    status STRING,
    record_count BIGINT,
    message STRING,
    start_time TIMESTAMP,
    end_time TIMESTAMP
)
USING DELTA;

##Appending logs in pipeline_logs table (Delta table)

In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import current_timestamp

def log_pipeline(pipeline, table, stage, status, count, message, start, end):

    log_row = [Row(
        pipeline_name=pipeline,
        table_name=table,
        stage=stage,
        status=status,
        record_count=count,
        message=message,
        start_time=start,
        end_time=end
    )]

    df = spark.createDataFrame(log_row)

    df.write.mode("append").saveAsTable("ecommerce.bronze.pipeline_logs")

##Dictionary of table_name with corresponding path

In [0]:
base_path = "/Volumes/ecommerce/bronze/raw_files/E-Commerce_Dataset_Kaggle/"

files = {
    "customers": "olist_customers_dataset.csv",
    "products": "olist_products_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "category": "product_category_name_translation.csv"
}

##Function to load csv file to the tables

In [0]:
from pyspark.sql.functions import current_timestamp, col
import datetime

def load_to_bronze(file_path, table_name):

    start_time = datetime.datetime.now()

    try:

        print(f"Starting load for {table_name}")

        df = (
            spark.read
            .option("header","true")
            .option("inferSchema","true")
            .csv(file_path)
        )

        record_count = df.count()

        df = (
            df
            .withColumn("ingestion_timestamp", current_timestamp())
            .withColumn("source_file", col("_metadata.file_path"))
        )

        (
            df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(f"ecommerce.bronze.{table_name}")
        )

        end_time = datetime.datetime.now()

        print(f"{table_name} loaded successfully")

        log_pipeline(
            "bronze_ingestion",
            table_name,
            "bronze_load",
            "SUCCESS",
            record_count,
            "Table loaded successfully",
            start_time,
            end_time
        )

    except Exception as e:

        end_time = datetime.datetime.now()

        print(f"Error loading {table_name}: {e}")

        log_pipeline(
            "bronze_ingestion",
            table_name,
            "bronze_load",
            "FAILED",
            0,
            str(e),
            start_time,
            end_time
        )

##Loading the .csv files to the tables

In [0]:
for table, file in files.items():
    path = f"{base_path}{file}"
    load_to_bronze(path, table)

##Validation and Testing of bronze layer

In [0]:
display(dbutils.fs.ls("/Volumes/ecommerce/bronze/raw_files/"))

In [0]:
spark.sql("""
SELECT *
FROM ecommerce.bronze.pipeline_logs
ORDER BY start_time DESC
""").show()

In [0]:
tables = [
"customers",
"products",
"orders",
"order_items",
"payments",
"reviews",
"sellers",
"geolocation",
"category",
"reviews_parsed"
]

for t in tables:

    count = spark.sql(f"SELECT COUNT(*) FROM ecommerce.bronze.{t}").collect()[0][0]

    print(f"{t}: {count} rows")

##Exporting the pipeline_logs table (Delta table) in the form of .logs(.txt) file

In [0]:
from pyspark.sql.functions import concat_ws

logs = spark.table("ecommerce.bronze.pipeline_logs")

formatted = logs.select(
    concat_ws(" | ",
        "start_time",
        "end_time",
        "pipeline_name",
        "table_name",
        "status",
        "record_count",
        "message"
    ).alias("value")
)

formatted.coalesce(1).write.mode("overwrite").text(
"/Volumes/ecommerce/bronze/export_logs/pipeline_logs.log"
)

In [0]:
from pyspark.sql.functions import current_timestamp, col
import datetime

def load_to_bronze(file_path, table_name):

    start_time = datetime.datetime.now()

    try:

        print(f"Starting load for {table_name}")

        df = (
            spark.read
            .format("csv")
            .option("header", "true")
            .option("inferSchema", "true")
            .option("multiLine", "true")      # handle multiline comments
            .option("quote", "\"")            # handle quoted text
            .option("escape", "\"")           # escape quotes inside text
            # .option("mode", "PERMISSIVE")     # don't fail pipeline
            .option("badRecordsPath", "/Volumes/ecommerce/bronze/bad_records/")
            .load(file_path)
        )

        record_count = df.count()

        df = (
            df
            .withColumn("ingestion_timestamp", current_timestamp())
            .withColumn("source_file", col("_metadata.file_path"))
        )

        (
            df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(f"ecommerce.bronze.{table_name}")
        )

        end_time = datetime.datetime.now()

        print(f"{table_name} loaded successfully")

        log_pipeline(
            "bronze_ingestion",
            table_name,
            "bronze_load",
            "SUCCESS",
            record_count,
            "Table loaded successfully",
            start_time,
            end_time
        )

    except Exception as e:

        end_time = datetime.datetime.now()

        print(f"Error loading {table_name}: {e}")

        log_pipeline(
            "bronze_ingestion",
            table_name,
            "bronze_load",
            "FAILED",
            0,
            str(e),
            start_time,
            end_time
        )

In [0]:
base_path = "/Volumes/ecommerce/bronze/raw_files/E-Commerce_Dataset_Kaggle/"

files_parsed = {  
    "reviews_parsed": "olist_order_reviews_dataset.csv"   
}

In [0]:
for table, file in files_parsed.items():
    path = f"{base_path}{file}"
    load_to_bronze(path, table)

In [0]:
bad_records = df.filter(col("_corrupt_record").isNotNull())

bad_records.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable(f"ecommerce.bronze.reviews_parsed_bad_records")

In [0]:
df = df.filter(col("_corrupt_record").isNull())

In [0]:
display(spark.sql("""
SELECT *
FROM ecommerce.bronze.reviews_parsed
"""))